# Experiment 1: The Recurring Concept Test
This experiment proves that Soft Reset strategy is superior to the Hard Reset used in standard ADWIN Bagging or Vanilla SRP. We will cycle the data through three stages: Concept A → Concept B → Concept A.

The Hypothesis: When Concept A returns, Vanilla SRP will have to relearn everything from scratch because it deleted its trees. C-DES will show a "fast recovery" because the trees (historical knowledge) were preserved, and only their competence scores need to be restored.

In [1]:
from river.datasets import synth
from river import metrics
from river.drift import ADWIN
from river.ensemble import SRPClassifier as VanillaSRPClassifier # Import original as alias
from src.streaming_random_patches import SRPClassifier

%load_ext autoreload
%autoreload 2

In [8]:
# 1. Create a cycling stream: A -> B -> A
# Function 0 and 2 are distinct; cycling back to 0 at instance 4000
stream_a1 = synth.Agrawal(classification_function=0, seed=42)
stream_b = synth.Agrawal(classification_function=2, seed=42)
stream_a2 = synth.Agrawal(classification_function=0, seed=42)

data_stream = []
data_stream.extend(list(stream_a1.take(2000)))
data_stream.extend(list(stream_b.take(2000)))
data_stream.extend(list(stream_a2.take(2000)))

# 2. Initialize Models
vanilla = VanillaSRPClassifier(n_models=10, drift_detector=ADWIN(), seed=42)
cdes = SRPClassifier(n_models=10, n_clusters=3, drift_detector=ADWIN(), seed=42)

# 3. Track Accuracy
v_acc = metrics.Accuracy()
c_acc = metrics.Accuracy()

In [9]:
print("Stage 1: Concept A | Stage 2: Concept B | Stage 3: Return to Concept A")
for i, (x, y) in enumerate(data_stream):
    # Predict and Update
    py_v = vanilla.predict_one(x)
    py_c = cdes.predict_one(x)
    if py_v is not None: v_acc.update(y, py_v)
    if py_c is not None: c_acc.update(y, py_c)
    
    # Train
    vanilla.learn_one(x, y)
    cdes.learn_one(x, y)
    
    if (i + 1) % 2000 == 0:
        print(f"Instance {i+1} Results -> Vanilla: {v_acc.get():.2%} | C-DES: {c_acc.get():.2%}")

Stage 1: Concept A | Stage 2: Concept B | Stage 3: Return to Concept A
Instance 2000 Results -> Vanilla: 98.40% | C-DES: 98.50%
Instance 4000 Results -> Vanilla: 77.07% | C-DES: 96.25%
Instance 6000 Results -> Vanilla: 73.95% | C-DES: 95.00%


# Experiment 2: Hyperparameter Sensitivity ($K$ Clusters)

This experiment addresses the "Context Mapping". We want to see how the number of clusters affects performance.

In [10]:
cluster_results = {}

for k in [1, 3, 5, 10]:
    # Reset stream for each test
    test_stream = synth.ConceptDriftStream(
        stream=synth.Agrawal(classification_function=0, seed=42),
        drift_stream=synth.Agrawal(classification_function=2, seed=42),
        position=2000, width=50, seed=123
    ).take(4000)
    
    model = SRPClassifier(n_models=10, n_clusters=k, drift_detector=ADWIN(), seed=42)
    acc = metrics.Accuracy()
    
    for x, y in test_stream:
        y_pred = model.predict_one(x)
        if y_pred is not None: acc.update(y, y_pred)
        model.learn_one(x, y)
    
    cluster_results[k] = acc.get()
    print(f"Final Accuracy for K={k}: {acc.get():.2%}")

Final Accuracy for K=1: 96.75%
Final Accuracy for K=3: 96.22%
Final Accuracy for K=5: 95.92%
Final Accuracy for K=10: 95.37%


# Experiment 3: Centroid-Based Drift (RandomRBF)

In this scenario, the "concepts" are physical locations in the feature space. We will simulate a drift where the centers of the data clusters shift.

In [11]:
print("--- 🏎️ STARTING RACE 3: The RandomRBF Centroid Drift Test ---")

# 1. Setup RandomRBFDrift - Designed for evaluating clustering 
# We start with one set of centroids and drift to another
stream = synth.ConceptDriftStream(
    stream=synth.RandomRBF(seed_model=42, seed_sample=42, n_classes=2),
    drift_stream=synth.RandomRBF(seed_model=99, seed_sample=42, n_classes=2),
    position=2000,
    width=50,
    seed=123
).take(4000)

# 2. Models
vanilla = VanillaSRPClassifier(n_models=10, drift_detector=ADWIN(), seed=42)
cdes = SRPClassifier(n_models=10, n_clusters=3, drift_detector=ADWIN(), seed=42)

# 3. Track Metrics (Using Balanced Accuracy and F1 as per your strategy [cite: 65])
v_metric = metrics.BalancedAccuracy() + metrics.F1()
c_metric = metrics.BalancedAccuracy() + metrics.F1()

--- 🏎️ STARTING RACE 3: The RandomRBF Centroid Drift Test ---


In [12]:
# 4. The Race Loop
for i, (x, y) in enumerate(stream):
    py_v = vanilla.predict_one(x)
    py_c = cdes.predict_one(x)
    
    if py_v is not None: v_metric.update(y, py_v)
    if py_c is not None: c_metric.update(y, py_c)
    
    vanilla.learn_one(x, y)
    cdes.learn_one(x, y)
    
    if (i + 1) % 1000 == 0:
        print(f"Instance {i+1} | Vanilla -> {v_metric} | C-DES -> {c_metric}")

Instance 1000 | Vanilla -> BalancedAccuracy: 81.11%
F1: 80.93% | C-DES -> BalancedAccuracy: 80.51%
F1: 80.28%
Instance 2000 | Vanilla -> BalancedAccuracy: 84.82%
F1: 84.36% | C-DES -> BalancedAccuracy: 84.56%
F1: 84.09%
Instance 3000 | Vanilla -> BalancedAccuracy: 85.29%
F1: 82.76% | C-DES -> BalancedAccuracy: 84.51%
F1: 81.84%
Instance 4000 | Vanilla -> BalancedAccuracy: 85.95%
F1: 82.54% | C-DES -> BalancedAccuracy: 85.82%
F1: 82.24%


In [13]:
print("\n" + "="*40)
print("🏆 FINAL RANDOM RBF RESULTS 🏆")
print("="*40)
print(f"VANILLA SRP: {v_metric}")
print(f"CUSTOM C-DES: {c_metric}")


🏆 FINAL RANDOM RBF RESULTS 🏆
VANILLA SRP: BalancedAccuracy: 85.95%
F1: 82.54%
CUSTOM C-DES: BalancedAccuracy: 85.82%
F1: 82.24%
